# VTIA: probar el tablero en Colab

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. Con GPU, anonimizar y analizar es mucho más rápido que en Streamlit Community Cloud.

Lo que pasa en este cuaderno:

1. Se clona el repositorio y se instalan las dependencias.
2. Se descarga un video de ejemplo y se prueba el conteo por línea de comandos.
3. Se abre el tablero de Streamlit y se publica con [bore](https://github.com/ekzhang/bore) para tener un enlace.
4. Al terminar, se apaga todo.

**Privacidad.** Cualquiera que tenga el enlace de bore puede abrir el tablero mientras esté encendido: no lo compartas y apágalo con la última celda. Todo video subido se anonimiza dentro del tablero antes de analizarse, pero el original sí pasa por los servidores de Google (fuera de Colombia). Para videos sensibles, anonimízalos antes en el PC servidor con `python -m vtia.cli anonymize`.

**Secreto** (icono de la llave, a la izquierda), solo si el repositorio es privado: `GITHUB_TOKEN`, con permiso de lectura.

## 1. Parámetros

In [ ]:
BRANCH = "feature/comparacion-modelos"   # rama con la anonimización de videos subidos
SAMPLE_URL = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"
PORT = 8501

## 2. Código y dependencias

In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import userdata

REPO = "/content/vtia"
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
auth = f"{token}@" if token else ""
url = f"https://{auth}github.com/alejodiazzz/Deteccion-y-Analisis-de-trafico-mediante-IA.git"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
%cd {REPO}

# torch/torchvision ya vienen con CUDA en Colab; requirements.txt no los fija.
# pandas se omite para no reemplazar el que Colab necesita (google-colab exige pandas 2.x);
# streamlit y altair funcionan con ese pandas.
reqs = [l for l in Path("requirements.txt").read_text().splitlines() if not l.strip().startswith("pandas")]
Path("/content/requirements-colab.txt").write_text("\n".join(reqs))
!pip install -q -r /content/requirements-colab.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay: Entorno de ejecución → GPU")

## 3. Video de ejemplo y prueba rápida

El video queda en `data/samples/`, así que aparece en el tablero en **«o elegir uno de data/»**. Es un video público de tráfico, sin personas identificables.

In [ ]:
!mkdir -p data/samples
!curl -sLo data/samples/vehicles.mp4 {SAMPLE_URL}
!python -m vtia.cli count data/samples/vehicles.mp4 --point configs/points/sample_vehicles.yaml --save-video

## 4. Abrir el tablero

Al final se imprime el enlace (`http://bore.pub:<puerto>`). La primera vez que se analiza con un modelo, Ultralytics descarga sus pesos.

Si el enlace no carga, revisa el registro con `!tail -30 /content/streamlit.log`.

In [ ]:
import re, threading, time, urllib.request

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "bore local"], check=False)

log = open("/content/streamlit.log", "w")
streamlit = subprocess.Popen(["python", "-m", "streamlit", "run", "app/dashboard.py", "--server.port", str(PORT)],
                             stdout=log, stderr=subprocess.STDOUT)

for _ in range(60):  # espera a que Streamlit responda (máx. 60 s)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit no arrancó: revisa /content/streamlit.log")

if not Path("/content/bore").exists():
    !curl -Ls https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz | tar xz -C /content
bore = subprocess.Popen(["/content/bore", "local", str(PORT), "--to", "bore.pub"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

link = []
def read_bore(stream):
    for line in stream:
        m = re.search(r"bore\.pub:(\d+)", line)
        if m and not link:
            link.append(f"http://bore.pub:{m.group(1)}")
threading.Thread(target=read_bore, args=(bore.stdout,), daemon=True).start()

for _ in range(20):
    if link:
        break
    time.sleep(0.5)
print("Tablero:", link[0] if link else "bore no respondió; vuelve a ejecutar esta celda")

## 5. Apagar

Cierra el enlace público. También se apaga solo si Colab se desconecta.

In [ ]:
for p in (bore, streamlit):
    p.terminate()
print("Tablero y túnel apagados.")